In [ ]:
import base64
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")

username = "AnikMallick"
repo_name = "bn-proverbs-in-context"

repo_dir = Path("/kaggle/working") / repo_name

auth = base64.b64encode(
    f"{username}:{token}".encode()
).decode()

try:
    subprocess.run(
        [
            "git",
            "-c",
            f"http.https://github.com/.extraheader="
            f"AUTHORIZATION: basic {auth}",
            "clone",
            f"https://github.com/{username}/{repo_name}.git",
            str(repo_dir),
        ],
        check=True,
    )
finally:
    del token
    del auth

print("Repository cloned successfully.")

In [ ]:
%cd /kaggle/working/bn-proverbs-in-context

!pip install -e .

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re, json, math, inspect
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import probad.builder as exp
from probad.schema import TestItem

BASE_ID = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER = "anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora"
DATA_PATH = Path("/kaggle/input/datasets/amallick0507/probad")
RESULTS_ROOT = Path("/kaggle/working/results")
RUNS = {"base": BASE_ID.split("/")[-1], "lora": BASE_ID.split("/")[-1] + "_lora"}
CONDITIONS = [("x1", "en"), ("x1", "bn"), ("x5", "en"), ("x5", "bn"),     # trained formats
              ("x2", "en"), ("x2", "bn"), ("x3", "en"), ("x3", "bn"),     # not trained
              ("x4", "en"), ("x4", "bn"), ("x7", "en"), ("x7", "bn")]
BATCH = 8
N_ITEMS = None 
SKIP_EXISTING = True
EXPERIMENTS = {"x1": exp.x1_correct_vs_literal, "x2": exp.x2_negative, "x3": exp.x3_none_of_these,
               "x4": exp.x4_meaning_no_context, "x5": exp.x5_three_way, "x7": exp.x7_no_question}
KEYS = ("correct", "literal", "plausible", "none")

In [3]:
ITEMS = [it.model_dump() for it in TestItem.load_items(DATA_PATH / "test.jsonl")]
ITEMS = ITEMS[:N_ITEMS] if N_ITEMS else ITEMS
print(len(ITEMS), "test items")

tok = AutoTokenizer.from_pretrained(BASE_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
import os
from probad.scoring import LoadBalancedModel

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")
os.environ['HF_TOKEN'] = secret_value_1
token = secret_value_1

adapter_path = snapshot_download(repo_id=ADAPTER, token=token)
print(adapter_path, os.listdir(adapter_path))  
adapter_path_v2 = adapter_path + "/anik-owl/qwen2.5-3b-instruct-bn-proverbs-lora/v2"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(model, adapter_path_v2).eval()
DEV = next(model.parameters()).device

def letter_id(L):
    ids = tok.encode(" " + L, add_special_tokens=False)
    return ids[1] if len(ids) > 1 and tok.decode(ids[:1]).strip() == "" else ids[0]
LID = {L: letter_id(L) for L in "ABC"}

600 test items


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
_norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().strip("।.!?\"'“”‘’ ").lower()

def build(fn, item, lang, seed):
    """-> (system, prompt, gold upper) or None. Builders without a seed argument have one order only."""
    ps = inspect.signature(fn).parameters
    if "seed" in ps or any(p.kind == p.VAR_KEYWORD for p in ps.values()):
        out = fn(item, lang=lang, seed=seed)
    elif seed == 0:
        out = fn(item, lang=lang)
    else:
        return None
    if out is None:
        return None
    out = tuple(out)
    if len(out) == 2:
        sy, pr, g = None, out[0], out[1]
    elif len(out) == 3:
        sy, pr, g = (None, out[0], out[1]) if isinstance(out[2], (list, tuple)) else out
    else:
        sy, pr, g = out[:3]
    return sy, pr, str(g).strip().upper()

def option_lines(prompt):
    block = prompt[prompt.rfind("\n\n"):] if "\n\n" in prompt else prompt
    return {m.group(1): m.group(2).strip() for m in re.finditer(r"^([ABC])\. (.*)$", block, flags=re.M)}

def option_key(text, item):
    """Which option a text is (correct / literal / plausible); anything else is 'none'."""
    cands = {"correct": [item["proverb"].get("meaning", {}).get(l) for l in ("en", "bn")], "literal": [], "plausible": []}
    for l in ("en", "bn"):
        o = item.get(l, {}).get("options", {})
        for k in ("correct", "literal", "plausible"):
            cands[k].append(o.get(k))
    t = _norm(text)
    for k, texts in cands.items():
        if any(c and _norm(c) == t for c in texts):
            return k
    for k, texts in cands.items():
        for c in texts:
            c = _norm(c) if c else ""
            if c and min(len(c), len(t)) >= 12 and (c in t or t in c):
                return k
    return "none"

def order_seeds(fn, item, lang):
    """Seed 0 + seeds that move the gold letter to every position (x3: gold stays C, so the other two swap)."""
    first = build(fn, item, lang, 0)
    if first is None:
        return []
    n = len(option_lines(first[1]))
    seeds, golds, prompts = [0], {first[2]}, {first[1]}
    for s in range(1, 300):
        b = build(fn, item, lang, s)
        if b and b[2] not in golds:
            seeds.append(s); golds.add(b[2]); prompts.add(b[1])
        if len(golds) >= n:
            break
    if len(seeds) == 1:
        for s in range(1, 300):
            b = build(fn, item, lang, s)
            if b and b[1] not in prompts:
                seeds.append(s); break
    return seeds

def to_text(prompt, system=None):
    """Chat template, then the prompt's answer cue at the start of the reply (same as training and scoring)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + prompt.rstrip().rsplit("\n", 1)[-1]

In [5]:
@torch.inference_mode()
def next_logprobs(texts):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEV)
    pos = (enc.attention_mask.cumsum(-1) - 1).clamp(min=0)          # correct positions with left padding
    out = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask, position_ids=pos, logits_to_keep=1)
    lp = torch.log_softmax(out.logits[:, -1].float(), dim=-1)
    if torch.isnan(lp).any():
        raise FloatingPointError("NaN log-probs (fp16 overflow?)")
    return lp

def score_texts(texts):
    """{text: (log-probs of A/B/C, top token)} for the unique texts, longest first (less padding)."""
    uniq = sorted(set(texts), key=len, reverse=True)
    res = {}
    for i in tqdm(range(0, len(uniq), BATCH), leave=False):
        b = uniq[i:i + BATCH]
        lp = next_logprobs(b)
        top = lp.argmax(-1).tolist()
        for t, row, tp in zip(b, lp, top):
            res[t] = ({L: row[LID[L]].item() for L in "ABC"}, tok.decode([tp]))
    return res

def score_row(item, prompt, gold, lps, top_token):
    lines = option_lines(prompt); letters = list(lines)
    keys = {L: option_key(t, item) for L, t in lines.items()}
    raw = {L: lps[L] for L in letters}
    mass = sum(math.exp(v) for v in raw.values())
    pred = max(raw, key=raw.get)
    row = {"gold": gold.lower(), "pred": pred.lower(), "correct": pred == gold, "n_opts": len(letters),
           "margin": raw[gold] - max(v for L, v in raw.items() if L != gold), "letter_mass": mass,
           "top_token": top_token, "top_is_letter": top_token.strip() in letters,
           "gold_key": keys[gold], "pred_key": keys[pred]}
    for L in "ABC":
        row[f"p_{L.lower()}"] = math.exp(raw[L]) / mass if L in raw else np.nan
        row[f"key_{L.lower()}"] = keys.get(L)
    for k in KEYS:
        row[f"p_{k}"] = sum(row[f"p_{L.lower()}"] for L in letters if keys[L] == k) if k in keys.values() else np.nan
    return row

def meta(run, e, lang, it, seed, order):
    m = it.get("meta", {}) or {}
    return {"model": BASE_ID + (" + LoRA" if run == "lora" else ""), "run": RUNS[run], "exp": e, "label": f"{e}_{lang}",
            "item_id": it["item_id"], "proverb_id": it["proverb"]["proverb_id"], "figurative": it["proverb"].get("figurative"),
            "split": m.get("split"), "subsets": "|".join(m.get("subsets") or []), "conv_lang": lang,
            "render": None, "prompt_lang": None, "condition": None, "seed": seed, "order": order}

In [7]:
# ================================================================ held-out negation wording (x2r), inference only
# Same item, same two options, same gold (the literal option) as x2. Only the question line changes:
#   EN: lower-case "not" and a different sentence shape;  BN: "নয়" instead of the trailing "না".
X2R_Q = {"en": "Which of these is not what {speaker} is saying?",
         "bn": "{speaker} যা বোঝাতে চাইছে, নিচের কোনটি তা নয়?"}
_QLINE = re.compile(r"^(Question:|প্রশ্ন:)\s*.*$", flags=re.M)

def x2_reworded(item, lang="en", seed=0):
    out = build(exp.x2_negative, item, lang, seed)
    if out is None:
        return None
    sy, pr, g = out
    spk = item[lang]["proverb_speaker"]
    new, n = _QLINE.subn(lambda m: f"{m.group(1)} {X2R_Q[lang].format(speaker=spk)}", pr, count=1)
    assert n == 1, f"question line not found:\n{pr[:300]}"
    return (sy, new, g) if sy else (new, g)

EXPERIMENTS["x2r"] = x2_reworded
CONDITIONS = [("x2r", "en"), ("x2r", "bn")]

# check one prompt per language before scoring
for l in ("en", "bn"):
    sy, pr, g = build(x2_reworded, ITEMS[0], l, 0)
    print(pr, "\n→ gold:", g, "\n" + "-" * 60)
print("OLD")
for l in ("en", "bn"):
    sy, pr, g = build(exp.x2_negative, ITEMS[0], l, 0)
    print(pr, "\n→ gold:", g, "\n" + "-" * 60)

Rina: Did you hear about Pintu? He put all his share-market profit back in, hoping to double it.
Sanjay: And the market crashed this morning. He lost everything. Oti lobhe tanti noshto.

Question: Which of these is not what Sanjay is saying?
A. Pintu lost everything because he greedily wanted even more profit.
B. A weaver was ruined because he wanted far too much for himself.
Answer: 
→ gold: B 
------------------------------------------------------------
রিনা: শুনেছ, পিন্টু শেয়ারের লাভের টাকা পুরোটাই আবার বাজারে ঢেলেছে, দ্বিগুণ করার আশায়?
সঞ্জয়: আর আজ সকালেই বাজার পড়ে গেল, সব শেষ। অতি লোভে তাঁতি নষ্ট।

প্রশ্ন: সঞ্জয় যা বোঝাতে চাইছে, নিচের কোনটি তা নয়?
A. আরও লাভের লোভ করতে গিয়েই পিন্টু নিজের সব খুইয়েছে।
B. বেশি চাওয়ার জন্য একজন কাপড় বোনার কারিগর সর্বস্ব হারিয়েছিল।
উত্তর: 
→ gold: B 
------------------------------------------------------------
OLD
Rina: Did you hear about Pintu? He put all his share-market profit back in, hoping to double it.
Sanjay: And the market crashed t

In [8]:
for e, lang in CONDITIONS:
    jobs = []
    for it in ITEMS:
        for order, s in enumerate(order_seeds(EXPERIMENTS[e], it, lang)):
            sy, pr, g = build(EXPERIMENTS[e], it, lang, s)
            jobs.append((it, s, order, pr, g, to_text(pr, sy)))
    for run in ("base", "lora"):
        path = RESULTS_ROOT / RUNS[run] / "raw" / f"{e}_{lang}.csv"
        if SKIP_EXISTING and path.exists():
            print(f"{run:4s} {e}_{lang}: already done"); continue
        path.parent.mkdir(parents=True, exist_ok=True)
        if run == "base":
            with model.disable_adapter():
                S = score_texts([j[5] for j in jobs])
        else:
            S = score_texts([j[5] for j in jobs])
        df = pd.DataFrame([{**meta(run, e, lang, it, s, order), **score_row(it, pr, g, *S[t])}
                           for it, s, order, pr, g, t in jobs])
        df.to_csv(path, index=False)
        acc = df.groupby("item_id")["correct"].mean()
        print(f"{run:4s} {e}_{lang}: acc seed0 {df[df.order == 0].correct.mean():.3f} | acc {acc.mean():.3f} | "
              f"letter mass {df.letter_mass.median():.3f} | {len(df)} rows")

  0%|          | 0/150 [00:00<?, ?it/s]

base x2r_en: acc seed0 0.807 | acc 0.815 | letter mass 0.999 | 1200 rows


  0%|          | 0/150 [00:00<?, ?it/s]

lora x2r_en: acc seed0 0.987 | acc 0.988 | letter mass 1.000 | 1200 rows


  0%|          | 0/150 [00:00<?, ?it/s]

base x2r_bn: acc seed0 0.488 | acc 0.476 | letter mass 0.996 | 1200 rows


  0%|          | 0/150 [00:00<?, ?it/s]

lora x2r_bn: acc seed0 0.940 | acc 0.937 | letter mass 0.999 | 1200 rows


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

FIG = RESULTS_ROOT / RUNS["lora"] / "figs"; FIG.mkdir(parents=True, exist_ok=True)
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
GAIN, LOSS = "#2a78d6", "#e34948"
LANG_COL = {"en": "#2a78d6", "bn": "#eb6834"}
TRAINED = {"x1", "x5", "x2"}
NAMES = {"x1": "x1 meaning (2 opts)", "x5": "x5 meaning (3 opts)", "x2": "x2 asked NOT", "x3": "x3 none-of-these",
         "x4": "x4 knowing (no context)", "x7": "x7 no question"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED})

def load(run):
    return pd.concat([pd.read_csv(p) for p in sorted((RESULTS_ROOT / RUNS[run] / "raw").glob("*.csv"))], ignore_index=True)

R = {run: load(run) for run in ("base", "lora")}

def items(df):
    """One row per item: accuracy over orders, same option in every order, mean letter mass,
    share of orders picking the figurative meaning (pred_key == 'correct')."""
    return df.groupby(["label", "exp", "conv_lang", "item_id", "proverb_id"]).agg(
        acc=("correct", "mean"), mass=("letter_mass", "mean"), fig=("pred_key", lambda s: (s == "correct").mean()),
        consistent=("pred_key", lambda s: s.nunique() == 1)).reset_index()

I = {run: items(df) for run, df in R.items()}

def paired(label, col="acc", n=2000, seed=0):
    """base -> LoRA on the same items: (base, lora, diff, lo, hi), 95% CI resampling proverbs."""
    a = I["base"].query("label == @label").set_index("item_id")
    b = I["lora"].query("label == @label").set_index("item_id")
    m = a[[col, "proverb_id"]].join(b[[col]], rsuffix="_l", how="inner")
    m["d"] = m[f"{col}_l"] - m[col]
    g = m.groupby("proverb_id")["d"].agg(["sum", "count"])
    pick = np.random.default_rng(seed).integers(0, len(g), (n, len(g)))
    boot = g["sum"].to_numpy()[pick].sum(1) / g["count"].to_numpy()[pick].sum(1)
    return m[col].mean(), m[f"{col}_l"].mean(), m["d"].mean(), *np.percentile(boot, [2.5, 97.5])

In [ ]:
rows = []
for label in sorted(set(I["base"].label) & set(I["lora"].label)):
    e, lang = label.split("_")
    b, l, d, lo, hi = paired(label)
    ib, il = I["base"].query("label == @label"), I["lora"].query("label == @label")
    m = ib.set_index("item_id")["acc"].to_frame("b").join(il.set_index("item_id")["acc"].rename("l"))
    rows.append({"task": e, "lang": lang, "trained": e in TRAINED, "base": b, "lora": l, "diff_pts": d * 100,
                 "ci_lo": lo * 100, "ci_hi": hi * 100, "fixed": int(((m.b < 0.5) & (m.l >= 0.5)).sum()),
                 "broken": int(((m.b >= 0.5) & (m.l < 0.5)).sum()),
                 "consistent_base": ib.consistent.mean(), "consistent_lora": il.consistent.mean(),
                 "mass_base": ib.mass.median(), "mass_lora": il.mass.median()})
TAB = pd.DataFrame(rows)
TAB["order"] = TAB.task.map({k: i for i, k in enumerate(NAMES)})
TAB = TAB.sort_values(["trained", "order", "lang"], ascending=[False, True, True]).drop(columns="order").reset_index(drop=True)
display(TAB.round(3))
TAB.to_csv(FIG.parent / "lora_vs_base.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 0.55 * TAB.task.nunique() + 1.6), sharey=True, gridspec_kw=dict(wspace=0.05))
tasks = list(dict.fromkeys(TAB.task))
for ax, lang in zip(axes, ["en", "bn"]):
    for y, t in enumerate(tasks):
        r = TAB[(TAB.task == t) & (TAB.lang == lang)]
        if r.empty:
            continue
        r = r.iloc[0]
        col = MUTED if abs(r.lora - r.base) < 0.005 else GAIN if r.lora > r.base else LOSS
        if abs(r.lora - r.base) > 0.03:
            ax.annotate("", xy=(r.lora, y), xytext=(r.base, y),
                        arrowprops=dict(arrowstyle="-|>", color=col, lw=2, shrinkA=5, shrinkB=5, mutation_scale=11))
        else:
            ax.plot([r.base, r.lora], [y, y], color=col, lw=2)
        ax.scatter(r.base, y, s=50, facecolor="white", edgecolor=MUTED, lw=1.6, zorder=3)
        ax.scatter(r.lora, y, s=50, color=col, edgecolor="white", lw=1, zorder=3)
        ax.text((r.base + r.lora) / 2, y - 0.32, f"{r.diff_pts:+.1f} [{r.ci_lo:+.1f}, {r.ci_hi:+.1f}]", ha="center",
                fontsize=7.5, color=INK, bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none"))
    n_tr = sum(t in TRAINED for t in tasks)
    ax.axhspan(-0.6, n_tr - 0.5, color=GAIN, alpha=0.05, zorder=0)       # shaded = trained formats
    ax.axvline(0.5, color=MUTED, ls="--", lw=1)
    ax.set_xlim(-0.03, 1.03); ax.set_xlabel("accuracy (mean over option orders)")
    ax.set_title(f"{lang} conversations", fontsize=10, loc="left", color=LANG_COL[lang])
axes[0].set_yticks(range(len(tasks)), [NAMES[t] + ("  (trained)" if t in TRAINED else "") for t in tasks])
axes[0].set_ylim(len(tasks) - 0.5, -0.8); axes[0].tick_params(axis="y", length=0)
fig.suptitle(f"{RUNS['base']}: before (open) → after LoRA (filled), on unseen test proverbs", x=0.01, ha="left",
             fontsize=10.5, color=INK, y=1.02)
fig.savefig(FIG / "L1_before_after.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# x1/x5/x7: the figurative meaning is right. x2: it is WRONG (asked what it does NOT mean).
# Understanding: x1 up while x2's figurative rate stays or falls. Shortcut: figurative rate up everywhere.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True, gridspec_kw=dict(wspace=0.05))
check = [t for t in ["x1", "x5", "x7", "x2"] if t in tasks]
for ax, lang in zip(axes, ["en", "bn"]):
    for y, t in enumerate(check):
        label = f"{t}_{lang}"
        if label not in set(I["base"].label):
            continue
        b, l, d, lo, hi = paired(label, col="fig")
        bad = (t == "x2" and d > 0) or (t != "x2" and d < 0)
        col = MUTED if abs(d) < 0.005 else LOSS if bad else GAIN
        if abs(l - b) > 0.03:
            ax.annotate("", xy=(l, y), xytext=(b, y),
                        arrowprops=dict(arrowstyle="-|>", color=col, lw=2, shrinkA=5, shrinkB=5, mutation_scale=11))
        else:
            ax.plot([b, l], [y, y], color=col, lw=2)
        ax.scatter(b, y, s=50, facecolor="white", edgecolor=MUTED, lw=1.6, zorder=3)
        ax.scatter(l, y, s=50, color=col, edgecolor="white", lw=1, zorder=3)
        ax.text((b + l) / 2, y - 0.34, f"{d * 100:+.1f} [{lo * 100:+.1f}, {hi * 100:+.1f}]", ha="center", fontsize=7.5,
                color=INK, bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none"))
    ax.axvline(0.5, color=MUTED, ls="--", lw=1); ax.set_xlim(-0.03, 1.03)
    ax.set_xlabel("share of answers picking the figurative meaning")
    ax.set_title(f"{lang} conversations", fontsize=10, loc="left", color=LANG_COL[lang])
axes[0].set_yticks(range(len(check)), [NAMES[t] + ("  ← figurative is wrong here" if t == "x2" else "") for t in check])
axes[0].set_ylim(len(check) - 0.5, -0.8); axes[0].tick_params(axis="y", length=0)
fig.suptitle("Shortcut check: red = moved the wrong way", x=0.01, ha="left", fontsize=10.5, color=INK, y=1.04)
fig.savefig(FIG / "L2_shortcut_check.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
gap = TAB.pivot(index="task", columns="lang", values=["base", "lora"])
gap = pd.DataFrame({"before": (gap["base"]["en"] - gap["base"]["bn"]) * 100,
                    "after": (gap["lora"]["en"] - gap["lora"]["bn"]) * 100}).reindex(tasks).dropna()
fig, ax = plt.subplots(figsize=(7, 0.5 * len(gap) + 1.2))
y = np.arange(len(gap))
ax.barh(y - 0.18, gap.before, height=0.34, color="#c9c8c2", label="before (base)")
ax.barh(y + 0.18, gap.after, height=0.34, color=LANG_COL["bn"], label="after LoRA")
for yi, (b, a) in enumerate(zip(gap.before, gap.after)):
    ax.text(max(a, 0) + 0.5, yi + 0.18, f"{a:+.1f}", va="center", fontsize=8, color=INK)
    ax.text(max(b, 0) + 0.5, yi - 0.18, f"{b:+.1f}", va="center", fontsize=8, color=MUTED)
ax.axvline(0, color=MUTED, lw=1)
ax.set_yticks(y, [NAMES[t] for t in gap.index]); ax.invert_yaxis(); ax.tick_params(axis="y", length=0)
ax.set_xlabel("English minus Bengali accuracy (points); smaller = gap closed")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title("Does fine-tuning close the Bengali gap?", fontsize=10, loc="left", color=INK)
fig.savefig(FIG / "L3_bengali_gap.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
if {"x4", "x1"} <= set(tasks):
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.6), sharey=True)
    for ax, lang in zip(axes, ["en", "bn"]):
        for run, ls, mk in [("base", "--", "o"), ("lora", "-", "s")]:
            k = TAB.query("task == 'x4' and lang == @lang")[run].iat[0]
            u = TAB.query("task == 'x1' and lang == @lang")[run].iat[0]
            ax.plot([0, 1], [k, u], ls=ls, marker=mk, color=LANG_COL[lang], lw=2, ms=7,
                    mfc="white" if run == "base" else LANG_COL[lang], label="before" if run == "base" else "after LoRA")
            ax.text(1.06, u, f"{u:.2f}", va="center", fontsize=8, color=INK)
            ax.text(-0.06, k, f"{k:.2f}", va="center", ha="right", fontsize=8, color=INK)
        ax.set_xticks([0, 1], ["knowing\n(x4, no context)", "using\n(x1, in conversation)"]); ax.set_xlim(-0.35, 1.3)
        ax.set_title({"en": "English", "bn": "Bengali"}[lang], fontsize=10, loc="left", color=LANG_COL[lang])
        ax.axhline(0.5, color=GRID, lw=1); ax.set_ylim(0, 1.05)
    axes[0].set_ylabel("accuracy"); axes[0].legend(frameon=False, fontsize=8, loc="lower right")
    fig.suptitle("Knowing vs using: x4 was never trained, so a rise there means real transfer", x=0.01, ha="left",
                 fontsize=10.5, color=INK, y=1.02)
    plt.tight_layout(); fig.savefig(FIG / "L4_knowing_vs_using.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
T = TAB.assign(name=TAB.task.map(NAMES) + " · " + TAB.lang)
fig, ax = plt.subplots(figsize=(7, 0.36 * len(T) + 1.2))
y = np.arange(len(T))
ax.barh(y, T.fixed, color=GAIN, height=0.62, label="fixed (wrong → right)")
ax.barh(y, -T.broken, color=LOSS, height=0.62, label="broken (right → wrong)")
for yi, (f, b) in enumerate(zip(T.fixed, T.broken)):
    ax.text(f + 1, yi, f"net {f - b:+d}", va="center", fontsize=8, color=INK)
ax.axvline(0, color=MUTED, lw=1)
ax.set_yticks(y, T.name); ax.invert_yaxis(); ax.tick_params(axis="y", length=0)
ax.set_xlabel("items (majority over option orders)")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title("What LoRA fixed and broke, per condition", fontsize=10, loc="left", color=INK)
fig.savefig(FIG / "L5_fixed_broken.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
def line(t, lang):
    r = TAB.query("task == @t and lang == @lang")
    return "n/a" if r.empty else (f"{r.base.iat[0]:.3f} → {r.lora.iat[0]:.3f} "
                                  f"({r.diff_pts.iat[0]:+.1f} pts [{r.ci_lo.iat[0]:+.1f}, {r.ci_hi.iat[0]:+.1f}])")
print(f"LoRA on {RUNS['base']} (trained on x1 + x5 of the train proverbs, tested on unseen proverbs)")
for t in tasks:
    print(f"  {NAMES[t]:28s} en {line(t, 'en')}   |   bn {line(t, 'bn')}")
print("  order consistency: base {:.0%} → LoRA {:.0%}".format(TAB.consistent_base.mean(), TAB.consistent_lora.mean()))
print("  Bengali gap (en − bn), mean over tasks: {:.1f} → {:.1f} pts".format(gap.before.mean(), gap.after.mean()))

In [9]:
!zip -r /kaggle/working/results-Qwen3B-finetune-results.zip /kaggle/working/results

  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct_lora/ (stored 0%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct_lora/raw/ (stored 0%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct_lora/raw/x2r_en.csv (deflated 82%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct_lora/raw/x2r_bn.csv (deflated 82%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct/ (stored 0%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct/raw/ (stored 0%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct/raw/x2r_en.csv (deflated 81%)
  adding: kaggle/working/results/Qwen2.5-3B-Instruct/raw/x2r_bn.csv (deflated 81%)
